In [51]:
import numpy as np
import torch
# Import Mamba# from mamba_ssm import Mamba
from M8bit2.cifar100.model import CIFAR100_Mamba



In [52]:

def quantize_weights(weights, scale_factor):
    """
    Quantize weights to int8 format.
    Args:
        weights (numpy.ndarray): Floating-point weights.
        scale_factor (float): Scale factor for quantization.
    Returns:
        Tuple: Quantized weights, scale factor
    """
    quantized_weights = np.clip(np.round(weights * scale_factor), -128, 127).astype(np.int8)
    return quantized_weights, scale_factor


In [53]:

def save_weights(weights, scale_factor, weight_file, scale_file):
    """
    Save weights and scale to text files.
    Args:
        weights (numpy.ndarray): Quantized weights.
        scale_factor (float): Scale factor.
        weight_file (str): File to save weights.
        scale_file (str): File to save scale factor.
    """
    # Save weights as plain text
    np.savetxt(f"{weight_file}", weights.flatten(), fmt='%d')  # Save as integers
    # Save the scale factor
    with open(f"{scale_file}", "w") as f:
        f.write(f"{scale_factor}\n")


In [54]:

# # Simulated model weights (replace with your actual model)
# model = torch.load("M8bit2/cifar100/cifar100_best_model.pth").to("cpu")

#load model to cpu
device = torch.device("cpu")
model = CIFAR100_Mamba(
    input_dim=3,
    d_model=128,
    d_state=64,
    d_conv=64,
    expand=2,
    num_classes=100,
    dropout_rate=0.1  # Increased dropout for regularization
).to(device)

model.eval()

#load weights
model.load_state_dict(torch.load("M8bit2/cifar100/cifar100_best_model.pth", weights_only=True))

# Iterate over model parameters
for name, weights in model.named_parameters():
    if name.startswith("mamba."):  # Filter for "mamba." weights
        # Remove the "mamba." prefix
        clean_name = name[len("mamba."):]

        # Convert weights to numpy and quantize
        weights_np = weights.detach().numpy()
        if weights_np.size > 0:  # Ensure the weight tensor is not empty
            scale_factor = 127.0 / np.max(np.abs(weights_np))
            quantized_weights, scale = quantize_weights(weights_np, scale_factor)
            # Add KWS-on-MAMBA/_EightBitMamba/weights to clean_name
            clean_name = f"_EightBitMamba/weights/{clean_name}"



            #  Save weights and scale factor with cleaned name
            save_weights(quantized_weights, scale, f"{clean_name}_weights.txt", f"{clean_name}_scale.txt")
            print(f"Saved {clean_name} weights and scale.")
        else:
            print(f"Skipped {name}: no data.")

Saved _EightBitMamba/weights/A_log weights and scale.
Saved _EightBitMamba/weights/D weights and scale.
Saved _EightBitMamba/weights/in_proj.weight weights and scale.
Saved _EightBitMamba/weights/conv1d.weight weights and scale.
Saved _EightBitMamba/weights/conv1d.bias weights and scale.
Saved _EightBitMamba/weights/x_proj.weight weights and scale.
Saved _EightBitMamba/weights/dt_proj.weight weights and scale.
Saved _EightBitMamba/weights/dt_proj.bias weights and scale.
Saved _EightBitMamba/weights/out_proj.weight weights and scale.
